# Module 02 · Regret Matching & No-Regret Learning

Run from the repo root after `pip install -e ".[dev]"`. Read the explanation first: `docs/modules/02_regret_matching.md`.

In [ ]:
import math
import matplotlib.pyplot as plt
from pokerbot.solvers.matrix_games import (
    RPS, BIASED_RPS, Learner, self_play, vs_fixed, row_utils, payoff_range, matrix_exploitability)
LABELS = ["Rock", "Paper", "Scissors"]

## 1. The worked example, reproduced
Opponent plays Rock, Scissors, Rock. Compare with the table in the docs page.

In [ ]:
L = Learner(3, "rm")
for opp in ([1, 0, 0], [0, 0, 1], [1, 0, 0]):
    s = L.strategy()
    u = row_utils(RPS, opp)
    value = sum(a * b for a, b in zip(s, u))
    L.update(s, u)
    print("sigma", [round(x, 3) for x in s], " u", u, " value", round(value, 3),
          " cumulative regret", [round(x, 3) for x in L.regret])
print("next sigma:", [round(x, 3) for x in L.strategy()])

## 2. Against a fixed opponent: lock onto the best response
Opponent plays Rock / Paper / Scissors = 0.5 / 0.3 / 0.2, so Paper is best (expected payoff 0.3).

In [ ]:
rec = vs_fixed(RPS, [0.5, 0.3, 0.2], 2000, "rm")
plt.figure(figsize=(7, 3.2))
for a in range(3):
    plt.plot(rec["t"], [s[a] for s in rec["avg_row"]], label=LABELS[a])
plt.xlabel("iteration"); plt.ylabel("average strategy"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 3. Self-play: the average converges, the current strategy does not

In [ ]:
rec = self_play(RPS, 5000, "rm")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for a in range(3):
    ax[0].plot(rec["t"], [s[a] for s in rec["cur_row"]], lw=0.7, label=LABELS[a])
    ax[1].plot(rec["t"], [s[a] for s in rec["avg_row"]], label=LABELS[a])
ax[0].set_title("CURRENT strategy (keeps swinging)"); ax[1].set_title("AVERAGE strategy (converges to 1/3)")
for x in ax: x.set_xlabel("iteration"); x.grid(alpha=.3)
ax[1].legend(); plt.show()

### The pitfall: start at the equilibrium and nothing ever moves

In [ ]:
stuck = self_play(RPS, 500, "rm", init_row=[1/3]*3, init_col=[1/3]*3)
print("max regret ever seen:", max(stuck["regret_row"]), " (zero: nothing is learned)")

## 4. Regret grows like sqrt(T): compare with the theoretical bound  Δ·sqrt(nT)

In [ ]:
T = 20000
rec = self_play(RPS, T, "rm", record_every=100)
bound = [payoff_range(RPS) * math.sqrt(3 * t) for t in rec["t"]]
plt.figure(figsize=(7, 3.4))
plt.plot(rec["t"], rec["regret_row"], label="row player's regret")
plt.plot(rec["t"], rec["regret_col"], label="column player's regret")
plt.plot(rec["t"], bound, "k--", label=r"bound  $\Delta\sqrt{nT}$")
plt.xlabel("iteration"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. Exploitability falls like 1/sqrt(T)
On a log-log plot that is a line of slope −½. The dotted line is exactly `1/sqrt(T)`.

In [ ]:
rec = self_play(RPS, 100000, "rm", record_every=100)
plt.figure(figsize=(6.5, 4))
plt.loglog(rec["t"], rec["exploitability"], label="exploitability of average strategy")
plt.loglog(rec["t"], [1 / math.sqrt(t) for t in rec["t"]], ":", label=r"$1/\sqrt{T}$")
plt.xlabel("iteration"); plt.legend(); plt.grid(True, which="both", alpha=.3); plt.show()
print("10x more accuracy needs about 100x more iterations.")

## 6. RM vs RM+ vs Hedge

In [ ]:
plt.figure(figsize=(7, 4))
for alg in ["rm", "rm+", "hedge"]:
    r = self_play(RPS, 20000, alg, record_every=100)
    plt.loglog(r["t"], r["exploitability"], label=alg)
plt.xlabel("iteration"); plt.ylabel("exploitability"); plt.legend(); plt.grid(True, which="both", alpha=.3); plt.show()

## 7. A game whose equilibrium is not uniform
Rock beats Scissors for 2 instead of 1. Solve `A x = 0` by hand and compare with what learning finds.

In [ ]:
rec = self_play(BIASED_RPS, 30000, "rm", record_every=30000)
print("learned:", [round(p, 3) for p in rec["avg_row"][-1]], "  by hand: [0.25, 0.5, 0.25]")
print("exploitability:", rec["exploitability"][-1])

## Try it yourself
- Change Rock-beats-Scissors to 3. Predict the equilibrium, then check.
- Make the opponent switch strategy halfway (write a loop with `Learner` directly). How long does RM take to adapt? RM+?
- Add linear averaging to `Learner.average_strategy`. Does RM+ improve?